In [1]:
import numpy

#1) 通过Apriori算法实现从交易记录中找到商品的频繁项集
def loadDataSet():
    return [[1, 3, 4], [2, 3, 5], [1, 2, 3, 5], [2, 5]]

In [2]:
# C1 是大小为1的所有候选项集的集合
def createC1(dataSet):
    C1 = []
    for transaction in dataSet:
        for item in transaction:
            if not [item] in C1:
                C1.append([item])  # 存储所有不重复的单个商品
    C1.sort()
    # 返回冻结集合（不可修改）的列表，便于作为字典的键
    return list(map(frozenset, C1))

In [3]:
# 该函数用于从 C1 生成 L1 。
def scanD(D, Ck, minSupport):
    # 参数：数据集、候选项集列表 Ck 以及感兴趣项集的最小支持度 minSupport
    ssCnt = {}  # 存储每个候选集的出现次数
    for tid in D:  # 遍历每条交易记录
        for can in Ck:  # 遍历每个候选集
            if can.issubset(tid):  # 若候选集是交易记录的子集（即该交易包含候选集）
                if can not in ssCnt:  # 修复Python3兼容问题（原has_key已移除）
                    ssCnt[can] = 1
                else:
                    ssCnt[can] += 1  # 计数+1
    numItems = float(len(D))  # 交易总条数
    retList = []  # 存储满足最小支持度的频繁项集（Lk）
    supportData = {}  # 存储所有候选集的支持度
    for key in ssCnt:
        support = ssCnt[key] / numItems  # 支持度 = 候选集出现次数 / 总交易数
        if support >= minSupport:
            retList.insert(0, key)  # 满足条件的项集加入Lk
        supportData[key] = support  # 记录所有项集的支持度
    return retList, supportData

In [4]:
# total apriori：组合，向上合并
def aprioriGen(Lk, k):  # 创建Ck，参数：频繁项集列表Lk与项集元素个数k
    retList = []
    lenLk = len(Lk)
    for i in range(lenLk):
        for j in range(i + 1, lenLk):  # 两两组合遍历Lk中的频繁项集
            # 取前k-2个元素（用于判断是否可合并）
            L1 = list(Lk[i])[:k-2]
            L2 = list(Lk[j])[:k-2]
            L1.sort()
            L2.sort()
            if L1 == L2:  # 若前k-2个元素相同，则合并两个项集（避免重复）
                retList.append(Lk[i] | Lk[j])  # 集合合并（生成k项集）
    return retList

In [5]:
def apriori(dataSet, minSupport=0.5):
    C1 = createC1(dataSet)  # 生成1项候选集
    D = list(map(set, dataSet))  # 将交易记录转为集合（便于子集判断）
    L1, supportData = scanD(D, C1, minSupport)  # 筛选1项频繁项集L1
    L = [L1]  # 存储所有频繁项集（L1, L2, ...）
    k = 2
    # 循环生成更大的频繁项集，直到没有新的项集生成
    while len(L[k-2]) > 0:  # L[k-2]是k-1项频繁项集，若为空则停止
        Ck = aprioriGen(L[k-2], k)  # 生成k项候选集
        Lk, supK = scanD(D, Ck, minSupport)  # 筛选k项频繁项集Lk
        supportData.update(supK)  # 更新支持度字典
        L.append(Lk)  # 将Lk加入结果列表
        k += 1
    return L, supportData

In [6]:
apriori(loadDataSet())[0]

[[frozenset({5}), frozenset({2}), frozenset({3}), frozenset({1})],
 [frozenset({2, 3}), frozenset({3, 5}), frozenset({2, 5}), frozenset({1, 3})],
 [frozenset({2, 3, 5})],
 []]

In [7]:
#（2）通过（1）中计算的频繁项集，挖掘关联规则
# 生成关联规则
def generateRules(L, supportData, minConf=0.7):
    # 参数：频繁项集列表、支持度字典、最小可信度阈值
    bigRuleList = []  # 存储所有关联规则
    # 从2项频繁项集开始（1项集无法生成规则）
    for i in range(1, len(L)):
        for freqSet in L[i]:  # 遍历每个k项频繁项集（k>=2）
            # 生成后件为单个元素的候选规则
            H1 = [frozenset([item]) for item in freqSet]
            if i > 1:  # 若项集元素数>2（即k>=3），需进一步合并后件
                rulesFromConseq(freqSet, H1, supportData, bigRuleList, minConf)
            else:  # 若为2项集，直接计算可信度
                calcConf(freqSet, H1, supportData, bigRuleList, minConf)
    return bigRuleList

In [8]:
# 生成候选规则集合：计算规则的可信度以及找到满足最小可信度要求的规则
def calcConf(freqSet, H, supportData, brl, minConf=0.7):
    # 针对项集中只有两个元素时，计算可信度
    prunedH = []  # 存储满足最小可信度的后件
    for conseq in H:  # 遍历后件候选集
        # 可信度 = 频繁项集支持度 / 前件支持度（前件 = 频繁项集 - 后件）
        conf = supportData[freqSet] / supportData[freqSet - conseq]
        if conf >= minConf:
            print(freqSet - conseq, '-->', conseq, 'conf:', conf)  # 输出规则
            brl.append( (freqSet - conseq, conseq, conf) )  # 保存规则
            prunedH.append(conseq)  # 保存满足条件的后件（用于后续合并）
    return prunedH

In [9]:
# 合并：生成后件包含多个元素的规则
def rulesFromConseq(freqSet, H, supportData, brl, minConf=0.7):
    # 参数：频繁项集、后件列表H、支持度字典、规则列表、最小可信度
    m = len(H[0])  # 当前后件中每个集合的元素数
    if len(freqSet) > (m + 1):  # 若频繁项集元素数 > 后件元素数 + 1（可进一步拆分）
        Hmp1 = aprioriGen(H, m + 1)  # 合并后件，生成元素数为m+1的新后件
        Hmp1 = calcConf(freqSet, Hmp1, supportData, brl, minConf)  # 计算新规则的可信度
        if len(Hmp1) > 1:  # 若有多个满足条件的后件，递归合并生成更多规则
            rulesFromConseq(freqSet, Hmp1, supportData, brl, minConf)

In [10]:
generateRules(apriori(loadDataSet())[0], apriori(loadDataSet())[1])

frozenset({5}) --> frozenset({2}) conf: 1.0
frozenset({2}) --> frozenset({5}) conf: 1.0
frozenset({1}) --> frozenset({3}) conf: 1.0


[(frozenset({5}), frozenset({2}), 1.0),
 (frozenset({2}), frozenset({5}), 1.0),
 (frozenset({1}), frozenset({3}), 1.0)]

In [ ]:
import numpy

#1) 通过Apriori算法实现从交易记录中找到商品的频繁项集
def loadDataSet():
    return [[1, 3, 4], [2, 3, 5], [1, 2, 3, 5], [2, 5]]

# C1 是大小为1的所有候选项集的集合
def createC1(dataSet):
    C1 = []
    for transaction in dataSet:
        for item in transaction:
            if not [item] in C1:
                C1.append([item]) #store all the item unrepeatedly
    C1.sort()
    #return map(frozenset, C1)#frozen set, user can't change it.
    return list(map(frozenset, C1))


#该函数用于从 C1 生成 L1 。
def scanD(D, Ck, minSupport):
    #参数：数据集、候选项集列表 Ck 以及感兴趣项集的最小支持度 minSupport
    ssCnt={}
    for tid in D:#遍历数据集
        for can in Ck:#遍历候选项
            if can.issubset(tid):#判断候选项中是否含数据集的各项
                if not ssCnt.has_key(can): # python3 can not support
                    ssCnt[can]=1 #不含设为1
                else: ssCnt[can]+=1#含有则计数加1
    numItems=float(len(D))#数据集大小
    retList = []#L1初始化
    supportData = {}#记录候选项中各个数据的支持度
    for key in ssCnt:
        support = ssCnt[key]/numItems#计算支持度
        if support >= minSupport:
            retList.insert(0,key)#满足条件加入L1中
        supportData[key] = support
    return retList, supportData

#total apriori
def aprioriGen(Lk, k): #组合，向上合并
    #creates Ck 参数：频繁项集列表 Lk 与项集元素个数 k
    retList = []
    lenLk = len(Lk)
    for i in range(lenLk):
        for j in range(i+1, lenLk): #两两组合遍历
            L1 = list(Lk[i])[:k-2]; L2 = list(Lk[j])[:k-2]
            L1.sort(); L2.sort()
            if L1==L2: #若两个集合的前k-2个项相同时,则将两个集合合并
                retList.append(Lk[i] | Lk[j]) #set union
    return retList

def apriori(dataSet, minSupport = 0.5):
    C1 = createC1(dataSet)
    D = list(map(set, dataSet)) #python3
    L1, supportData = scanD(D, C1, minSupport)#单项最小支持度判断 0.5，生成L1
    L = [L1]
    k = 2
    while (len(L[k-2]) > 0):#创建包含更大项集的更大列表,直到下一个大的项集为空
        Ck = aprioriGen(L[k-2], k)#Ck
        Lk, supK = scanD(D, Ck, minSupport)#get Lk
        supportData.update(supK)
        L.append(Lk)
        k += 1
    return L, supportData


apriori(loadDataSet())[0]

#（2）通过（1）中计算的频繁项集，挖掘关联规则
#生成关联规则
def generateRules(L, supportData, minConf=0.7):
    #频繁项集列表、包含那些频繁项集支持数据的字典、最小可信度阈值
    bigRuleList = [] #存储所有的关联规则
    for i in range(1, len(L)): #只获取有两个或者更多集合的项目，从L1即第二个元素开始
        #两个及以上的可能有关联规则，单个元素的项集不存在关联问题
        for freqSet in L[i]:
            H1 = [frozenset([item]) for item in freqSet]
            #该函数遍历L中的每一个频繁项集并对每个频繁项集创建只包含单个元素集合的列表
            if (i > 1):
                #如果频繁项集元素数目超过2,那么会考虑对它做进一步的合并
                rulesFromConseq(freqSet, H1, supportData, bigRuleList, minConf)
            else:
                #一层时，后件数为1
                calcConf(freqSet, H1, supportData, bigRuleList, minConf)# 调用函数2
    return bigRuleList


#生成候选规则集合：计算规则的可信度以及找到满足最小可信度要求的规则
def calcConf(freqSet, H, supportData, brl, minConf=0.7):
    #针对项集中只有两个元素时，计算可信度
    prunedH = [] #返回一个满足最小可信度要求的规则列表
    for conseq in H:#H后件，遍历H中的所有项集并计算它们的可信度值
        conf = supportData[freqSet]/supportData[freqSet-conseq] #可信度计算，结合支持
        if conf >= minConf:
            print(freqSet-conseq, '-->', conseq, 'conf:', conf)
            #如果某条规则满足最小可信度值,那么将这些规则输出到屏幕显示
            brl.append( (freqSet-conseq, conseq, conf) )#添加到规则里，brl 是前面通过检
            prunedH.append(conseq)#同样需要放入列表后面检查
    return prunedH


#合并
def rulesFromConseq(freqSet, H, supportData, brl, minConf=0.7):
    #参数:一个是频繁项集,另一个是可以出现在规则右部的元素列表 H
    m = len(H[0])
    if (len(freqSet) > (m + 1)): #频繁项集元素数目大于单个集合的元素数
        Hmp1 = aprioriGen(H, m+1)#存在不同顺序、元素相同的集合，合并具有相同部分的集合
        Hmp1 = calcConf(freqSet, Hmp1, supportData, brl, minConf)#计算可信度
        if (len(Hmp1) > 1):
            #满足最小可信度要求的规则列表多于1,则递归来到判断是否可以进一步组合这些规则
            rulesFromConseq(freqSet, Hmp1, supportData, brl, minConf)


generateRules(apriori(loadDataSet())[0],apriori(loadDataSet())[1])